In [1]:
from ortools.sat.python import cp_model
from datetime import datetime
import pandas as pd
import re
import shutil
from pathlib import Path

# ==============================
# PATHS
# ==============================

BASE_DIR = Path().resolve().parent
DATA_DIR = BASE_DIR / "data"

CHAMADOS_PATH = DATA_DIR / "chamados.xlsx"
LIBERACAO_PATH = DATA_DIR / "liberacao_tecnicos_clientes.xlsx"
AGENDA_PATH = DATA_DIR / "agenda.xlsx"
VERIFICACAO_PATH = DATA_DIR / "chamados_verificacao.xlsx"
NAO_AGENDADOS_PATH = DATA_DIR / "chamados_nao_agendados.xlsx"


# ==============================
# NORMALIZAÇÃO
# ==============================

def normalizar_texto(s):
    return (
        str(s)
        .strip()
        .lower()
        .replace("não", "nao")
        .replace("nâo", "nao")
        .replace("nan", "nao")
    )


# ==============================
# BASE
# ==============================

def extrair_numero(s):
    return int(re.search(r'\d+', str(s)).group())


def carregar_dados():
    return (
        pd.read_excel(CHAMADOS_PATH),
        pd.read_excel(LIBERACAO_PATH)
    )


def atualizar_chamados_excel(df, path):
    df.reset_index(drop=True, inplace=True)
    df.to_excel(path, index=False)


def obter_todos_tecnicos(lib_df):
    return list(lib_df.columns.drop('cliente'))


# ==============================
# AGENDAS
# ==============================

def carregar_agendas(tecnicos):
    xls = pd.ExcelFile(AGENDA_PATH)
    agendas = {}

    for t in tecnicos:
        if t in xls.sheet_names:
            df = pd.read_excel(xls, sheet_name=t)

            df['data'] = pd.to_datetime(df['data'], errors='coerce')

            df['agenda_efetivada'] = df['agenda_efetivada'].apply(normalizar_texto)
            df['status_agenda'] = df['status_agenda'].apply(normalizar_texto)

            df['id'] = df['id'].fillna('').astype(str).str.strip()

            agendas[t] = df

    return agendas


# ==============================
# LIBERAÇÃO
# ==============================

def tecnico_liberado_para_cliente(tecnico, cliente, lib_df):
    try:
        val = lib_df.loc[lib_df['cliente'] == cliente, tecnico].values
        return len(val) > 0 and str(val[0]).lower() == 'liberado'
    except:
        return False


# ==============================
# SOLVER (INALTERADO)
# ==============================

def alocar_chamado(chamado, agendas, tecnicos, lib_df, alocacoes_cliente_dia):

    model = cp_model.CpModel()
    tempo = int(chamado['tempo_necessario'])

    possibilidades = []
    mapa = {}
    slots_map = {}

    for t in tecnicos:

        if not tecnico_liberado_para_cliente(t, chamado['cliente'], lib_df):
            continue

        if t not in agendas:
            continue

        agenda = agendas[t]
        datas = sorted(agenda['data'].dt.normalize().dropna().unique())

        for d in datas:

            if (chamado['cliente'], d.date()) in alocacoes_cliente_dia:
                continue

            slots = agenda[
                (agenda['data'].dt.normalize() == d) &
                (agenda['agenda_efetivada'] == 'nao') &
                (agenda['id'] == '')
            ]

            if len(slots) >= tempo:
                var = model.NewBoolVar(f"{t}_{d}")
                possibilidades.append((var, d.timestamp()))
                mapa[(t, d)] = var
                slots_map[(t, d)] = len(slots) - tempo

    if not possibilidades:
        return None, None, None

    model.AddExactlyOne(v for v, _ in possibilidades)
    model.Minimize(sum(ts * v for v, ts in possibilidades))

    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    if status in [cp_model.OPTIMAL, cp_model.FEASIBLE]:
        for (t, d), v in mapa.items():
            if solver.BooleanValue(v):
                return t, d, slots_map[(t, d)]

    return None, None, None


# ==============================
# FUNÇÕES DE AGENDA (RESTAURADAS)
# ==============================

def montar_df_alocacoes(tecnico, data, chamados,
                        adicionar_deslocamento=False, slots_totais=None):

    linhas = []
    total = 0

    for c in chamados:
        tempo = int(c['tempo_necessario'])
        total += tempo

        for _ in range(tempo):
            linhas.append({
                'tecnico': tecnico,
                'data': data,
                'id': c['id'],
                'cliente': c['cliente'],
                'maquina': c.get('maquina', ''),
                'tipo_manutencao': c.get('tipo_manutencao', ''),
                'tempo_necessario': tempo,
                'prioridade': c.get('prioridade', ''),
                'data_abertura': c.get('data_abertura', ''),
                'agenda_efetivada': 'sugerida',
                'status_agenda': 'liberada'
            })

    if adicionar_deslocamento and slots_totais is not None:
        if slots_totais - total >= 1:
            linhas.append({
                'tecnico': tecnico,
                'data': data,
                'id': 0,
                'cliente': 'deslocamento',
                'maquina': '',
                'tipo_manutencao': '',
                'tempo_necessario': 1,
                'prioridade': '',
                'data_abertura': '',
                'agenda_efetivada': 'sugerida',
                'status_agenda': 'liberada'
            })

    return pd.DataFrame(linhas)


def atualizar_agenda_com_df(tecnico, data, df_alocacoes, agendas):

    agenda = agendas[tecnico]

    if isinstance(data, pd.Timestamp):
        data = data.date()

    mask = (
        (agenda['data'].dt.date == data) &
        (agenda['status_agenda'] == 'liberada') &
        (agenda['agenda_efetivada'] == 'nao') &
        (agenda['id'] == '')
    )

    slots = agenda.loc[mask]

    if len(slots) < len(df_alocacoes):
        return False

    for i, idx in enumerate(slots.index[:len(df_alocacoes)]):
        linha = df_alocacoes.iloc[i]

        agenda.at[idx, 'id'] = linha['id']
        agenda.at[idx, 'cliente'] = linha['cliente']
        agenda.at[idx, 'maquina'] = linha['maquina']
        agenda.at[idx, 'tipo_manutencao'] = linha['tipo_manutencao']
        agenda.at[idx, 'tempo_necessario'] = linha['tempo_necessario']
        agenda.at[idx, 'prioridade'] = linha['prioridade']
        agenda.at[idx, 'data_abertura'] = linha['data_abertura']
        agenda.at[idx, 'agenda_efetivada'] = 'sugerida'

    agendas[tecnico] = agenda
    return True


def salvar_agendas_no_excel(agendas, arquivo=AGENDA_PATH):

    with pd.ExcelWriter(arquivo, engine='openpyxl', mode='w') as writer:
        for t, df in agendas.items():
            df.to_excel(writer, sheet_name=t, index=False)


# ==============================
# PIPELINE
# ==============================

print("Iniciando pipeline...")

if CHAMADOS_PATH.exists():
    shutil.copyfile(CHAMADOS_PATH, VERIFICACAO_PATH)

alocacoes_cliente_dia = {}

while True:

    chamados_df, lib_df = carregar_dados()

    print("\nChamados restantes:", len(chamados_df))

    if chamados_df.empty:
        break

    tecnicos = obter_todos_tecnicos(lib_df)

    agendas = carregar_agendas(tecnicos)

    chamado = chamados_df.iloc[0]

    tecnico, data, slots = alocar_chamado(
        chamado,
        agendas,
        tecnicos,
        lib_df,
        alocacoes_cliente_dia
    )

    print("Resultado:", tecnico, data, slots)

    if not tecnico:
        chamados_df = chamados_df.iloc[1:]
        atualizar_chamados_excel(chamados_df, CHAMADOS_PATH)
        continue

    print(f"Alocado {chamado['id']} -> {tecnico} em {data}")

    df_alocacoes = montar_df_alocacoes(
        tecnico, data, [chamado],
        adicionar_deslocamento=False,
        slots_totais=slots
    )

    atualizar_agenda_com_df(tecnico, data, df_alocacoes, agendas)
    salvar_agendas_no_excel(agendas)

    alocacoes_cliente_dia[(chamado['cliente'], data.date())] = tecnico

    chamados_df = chamados_df.iloc[1:]
    atualizar_chamados_excel(chamados_df, CHAMADOS_PATH)

print("Pipeline finalizado.")

Iniciando pipeline...

Chamados restantes: 200
Resultado: tecnico_1 2026-01-01 00:00:00 8
Alocado 72 -> tecnico_1 em 2026-01-01 00:00:00


/tmp/ipykernel_36812/694090252.py:226: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'cliente_22' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  agenda.at[idx, 'cliente'] = linha['cliente']
/tmp/ipykernel_36812/694090252.py:227: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'maquina_5' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  agenda.at[idx, 'maquina'] = linha['maquina']
/tmp/ipykernel_36812/694090252.py:228: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'avaliacao_do_equipamento' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  agenda.at[idx, 'tipo_manutencao'] = linha['tipo_manutencao']
/tmp/ipykernel_36812/69


Chamados restantes: 199
Resultado: tecnico_1 2026-01-01 00:00:00 7
Alocado 9 -> tecnico_1 em 2026-01-01 00:00:00

Chamados restantes: 198
Resultado: tecnico_1 2026-01-01 00:00:00 5
Alocado 128 -> tecnico_1 em 2026-01-01 00:00:00

Chamados restantes: 197
Resultado: tecnico_4 2026-01-01 00:00:00 7
Alocado 18 -> tecnico_4 em 2026-01-01 00:00:00


/tmp/ipykernel_36812/694090252.py:226: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'cliente_6' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  agenda.at[idx, 'cliente'] = linha['cliente']
/tmp/ipykernel_36812/694090252.py:227: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'maquina_6' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  agenda.at[idx, 'maquina'] = linha['maquina']
/tmp/ipykernel_36812/694090252.py:228: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'manutencao_preventiva' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  agenda.at[idx, 'tipo_manutencao'] = linha['tipo_manutencao']
/tmp/ipykernel_36812/694090


Chamados restantes: 196
Resultado: tecnico_1 2026-01-01 00:00:00 4
Alocado 3 -> tecnico_1 em 2026-01-01 00:00:00

Chamados restantes: 195
Resultado: tecnico_1 2026-01-02 00:00:00 8
Alocado 151 -> tecnico_1 em 2026-01-02 00:00:00

Chamados restantes: 194
Resultado: tecnico_2 2026-01-01 00:00:00 8
Alocado 34 -> tecnico_2 em 2026-01-01 00:00:00


...


Chamados restantes: 193
Resultado: tecnico_2 2026-01-01 00:00:00 7
Alocado 153 -> tecnico_2 em 2026-01-01 00:00:00

Chamados restantes: 192
Resultado: tecnico_3 2026-01-01 00:00:00 8
Alocado 139 -> tecnico_3 em 2026-01-01 00:00:00


/tmp/ipykernel_36812/694090252.py:226: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'cliente_13' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  agenda.at[idx, 'cliente'] = linha['cliente']
/tmp/ipykernel_36812/694090252.py:227: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'maquina_6' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  agenda.at[idx, 'maquina'] = linha['maquina']
/tmp/ipykernel_36812/694090252.py:228: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'avaliacao_do_equipamento' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  agenda.at[idx, 'tipo_manutencao'] = linha['tipo_manutencao']
/tmp/ipykernel_36812/69


Chamados restantes: 191
Resultado: tecnico_1 2026-01-01 00:00:00 2
Alocado 190 -> tecnico_1 em 2026-01-01 00:00:00

Chamados restantes: 190
Resultado: tecnico_3 2026-01-02 00:00:00 6
Alocado 140 -> tecnico_3 em 2026-01-02 00:00:00

Chamados restantes: 189
Resultado: tecnico_2 2026-01-01 00:00:00 4
Alocado 71 -> tecnico_2 em 2026-01-01 00:00:00

Chamados restantes: 188
Resultado: tecnico_1 2026-01-01 00:00:00 0
Alocado 35 -> tecnico_1 em 2026-01-01 00:00:00

Chamados restantes: 187
Resultado: tecnico_2 2026-01-01 00:00:00 1
Alocado 186 -> tecnico_2 em 2026-01-01 00:00:00

Chamados restantes: 186
Resultado: tecnico_4 2026-01-01 00:00:00 6
Alocado 133 -> tecnico_4 em 2026-01-01 00:00:00

Chamados restantes: 185
Resultado: tecnico_1 2026-01-03 00:00:00 6
Alocado 82 -> tecnico_1 em 2026-01-03 00:00:00

Chamados restantes: 184
Resultado: tecnico_1 2026-01-04 00:00:00 6
Alocado 121 -> tecnico_1 em 2026-01-04 00:00:00

Chamados restantes: 183
Resultado: tecnico_3 2026-01-01 00:00:00 5
Alocado

/tmp/ipykernel_36812/694090252.py:226: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'cliente_27' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  agenda.at[idx, 'cliente'] = linha['cliente']
/tmp/ipykernel_36812/694090252.py:227: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'maquina_9' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  agenda.at[idx, 'maquina'] = linha['maquina']
/tmp/ipykernel_36812/694090252.py:228: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'manutencao_corretiva' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  agenda.at[idx, 'tipo_manutencao'] = linha['tipo_manutencao']
/tmp/ipykernel_36812/694090


Chamados restantes: 176
Resultado: tecnico_2 2026-01-02 00:00:00 6
Alocado 54 -> tecnico_2 em 2026-01-02 00:00:00

Chamados restantes: 175
Resultado: tecnico_3 2026-01-01 00:00:00 0
Alocado 146 -> tecnico_3 em 2026-01-01 00:00:00

Chamados restantes: 174
Resultado: tecnico_3 2026-01-02 00:00:00 5
Alocado 105 -> tecnico_3 em 2026-01-02 00:00:00

Chamados restantes: 173
Resultado: tecnico_2 2026-01-02 00:00:00 3
Alocado 45 -> tecnico_2 em 2026-01-02 00:00:00

Chamados restantes: 172
Resultado: tecnico_1 2026-01-02 00:00:00 0
Alocado 75 -> tecnico_1 em 2026-01-02 00:00:00

Chamados restantes: 171
Resultado: tecnico_2 2026-01-02 00:00:00 1
Alocado 23 -> tecnico_2 em 2026-01-02 00:00:00

Chamados restantes: 170
Resultado: tecnico_2 2026-01-02 00:00:00 0
Alocado 13 -> tecnico_2 em 2026-01-02 00:00:00

Chamados restantes: 169
Resultado: tecnico_1 2026-01-03 00:00:00 6
Alocado 104 -> tecnico_1 em 2026-01-03 00:00:00

Chamados restantes: 168
Resultado: tecnico_3 2026-01-02 00:00:00 4
Alocado 9